In [1]:
from h5dataset import h5set
import os
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f'Selected device: {device}')

Selected device: cuda


In [2]:
dir_path = "/mnt/data/train_test_val"
h5_path = "../data/H1_rechunked.h5"
dir_ = os.listdir(dir_path)
if len(dir_)==0:
    noise = h5set(path=h5_path, dataset='noise')
    train, val, test = torch.utils.data.random_split(noise, [225000, 75000, 25000])
    injection = h5set(path=h5_path, dataset='injection')
    test = ConcatDataset([test, injection])
    torch.save(train, "/mnt/data/train_test_val/train.pt") #just saves indices
    torch.save(val, "/mnt/data/train_test_val/val.pt")
    torch.save(test, "/mnt/data/train_test_val/test.pt")
else:
    train = torch.load("/mnt/data/train_test_val/train.pt", weights_only=False)
    val = torch.load("/mnt/data/train_test_val/val.pt", weights_only=False)
    test = torch.load("/mnt/data/train_test_val/test.pt", weights_only=False)


training = DataLoader(train,
                batch_size=1,
                shuffle=True,
                num_workers=0,
                #persistent_workers=True,
                drop_last=True,
                )


In [3]:
class Encoder_Moreno(nn.Module):
    def __init__(self, sq_len, num_feat, exp_dim, compr_dim, num_layers, v=False):
        super().__init__()
        ## Useful quantities
        self.v = v
        self.sq_len = sq_len
        self.El1 = nn.LSTM(input_size=num_feat, 
                           hidden_size=exp_dim,
                           num_layers=num_layers,
                           batch_first=True)
        self.El2 = nn.LSTM(input_size=exp_dim,
                           hidden_size=compr_dim, 
                           num_layers=num_layers,
                           batch_first=True)

    
    def forward(self, item):
        if self.v: 
            print(item.shape, "Input shape")
            item, h_c = self.El1(item)
            print(item.shape, "1st encoder layer output shape")
            item, h_c = self.El2(item)
            print(item.shape, "2nd encoder layer output shape")
            item = item[:,-1,:]
            print(item.shape, "return sequence= False analog")
            item = item.repeat(1, self.sq_len, 1)
            print(item.shape, "repeat vector 100x")
            return item
        else:
            item, h_c = self.El1(item)
            item, h_c = self.El2(item)
            item = item[:,-1,:]
            item = item.repeat(1, self.sq_len,1)
            return item



class Decoder_Moreno(nn.Module):
    def __init__(self, sq_len, num_feat, exp_dim, compr_dim, num_layers, v=False):
        super().__init__()
        ## Useful quantities
        self.sq_len = sq_len
        self.Dl1 = nn.LSTM(input_size=compr_dim, 
                           hidden_size=compr_dim,
                           num_layers=num_layers,
                           batch_first=True)
        self.Dl2 = nn.LSTM(input_size=compr_dim,
                           hidden_size=exp_dim,
                           num_layers=num_layers,
                           batch_first=True)
        self.TimeDistributed = nn.Conv1d(exp_dim,
                                        num_feat,
                                        kernel_size=1)

        
    def forward(self, item, v=False):
        if v: 
            item, h_c = self.Dl1(item)
            print(item.shape, "1st decoder layer output shape")
            item, h_c = self.Dl2(item)
            print(item.shape, "2nd decoder layer output shape")
            item = torch.movedim(item, 1,2)
            print(item.shape, "move dim shape")
            item = self.TimeDistributed(item)
            print(item.shape, "conv1d shape (time distributed)")
            return item
        else:
            item, _ = self.Dl1(item)
            item, _ = self.Dl2(item)
            item = torch.movedim(item, 1,2)
            item = self.TimeDistributed(item)
            return item



class AEric(nn.Module):
    def __init__(self,sq_len, num_feat, exp_dim, compr_dim, num_layers, v=False):
        super().__init__()
        self.Encoder = Encoder_Moreno(sq_len, num_feat, exp_dim, compr_dim, num_layers, v=False)
        self.Decoder = Decoder_Moreno(sq_len, num_feat, exp_dim, compr_dim, num_layers, v=False)

    
    def forward(self, item):
        encoded = self.Encoder(item)
        decoded = self.Decoder(encoded)
        return decoded

In [4]:
AE = AEric(sq_len=100,
            num_feat=1,
            exp_dim=32,
            compr_dim=8,
            num_layers=3)
loss_func = torch.nn.MSELoss()
lr = 5e-4
params_to_optimize = [{'params': AE.parameters()}]
optim = torch.optim.Adam(params_to_optimize,
                        lr=lr,
                        weight_decay=1e-5
                        )

AE.to(device)


AEric(
  (Encoder): Encoder_Moreno(
    (El1): LSTM(1, 32, batch_first=True)
    (El2): LSTM(32, 8, batch_first=True)
  )
  (Decoder): Decoder_Moreno(
    (Dl1): LSTM(8, 8, batch_first=True)
    (Dl2): LSTM(8, 32, batch_first=True)
    (TimeDistributed): Conv1d(32, 1, kernel_size=(1,), stride=(1,))
  )
)

In [5]:
def train_epoch(ae, device, dataloader,timestep, loss_fn, optim):
    ae.train()
    losses = []
    for batch_data in dataloader:
        num_time_steps = batch_data.shape[1]
        remainder = num_time_steps % timestep
        batch_data = batch_data[:, :-remainder]
        for sample_idx in range(batch_data.shape[0]):
            sequence = batch_data[sample_idx, :]
            segments = sequence.reshape(-1, timestep, 1)
            for segment_idx in range(segments.shape[0]):
                current_window = segments[segment_idx, :, :]
                c_w = current_window.unsqueeze(0)
                c_w = c_w.to(device)
                ae_output = ae(c_w)
                loss = loss_fn(ae_output,c_w)
                print(loss)
                optim.zero_grad()
                loss.backward()
                optim.step()

                losses.append(loss.detach().cpu().numpy())
    losses = np.mean(losses)
    return losses

In [6]:
num_epochs = 10
for epoch in range(num_epochs):
    print('EPOCH %d/%d' % (epoch + 1, num_epochs))
    ### Training (use the training function)
    train_loss = train_epoch(
        ae=AE,
        device=device,
        dataloader=training,
        timestep=100,
        loss_fn=loss_func,
        optim=optim)
    print(f'TRAIN - EPOCH {epoch+1}/{num_epochs} - loss: {train_loss}')

EPOCH 1/10


/mnt/miniconda3/envs/killgg/lib/python3.12/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([1, 100, 1])) that is different to the input size (torch.Size([1, 1, 100])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


tensor(0.2630, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.2423, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.2342, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.2472, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.2335, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.2088, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.2165, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.2005, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.1858, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.1976, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.1778, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.1708, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.1854, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.1693, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.1571, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.1563, device='cuda:0', grad_fn=<MseLossBackward0>)
tensor(0.1546, device='cuda:0', grad_fn=

KeyboardInterrupt: 